# PGC Explorer — Data Pipeline
Load all 12 PGC disorder datasets from HuggingFace, compute cross-disorder correlations, and export JSON snapshots for the web frontend.

**Requirements:** `pip install -r requirements.txt`

**Runtime:** ~30-60 minutes depending on network speed (downloads ~50GB of parquet data).

In [ ]:
import sys
sys.path.insert(0, "..")

from pgc_explorer.config import DISORDERS, DATA_DIR
from pgc_explorer.loader import load_disorder
from pgc_explorer.analysis import compute_correlation_matrix
from pgc_explorer.export import (
    export_manhattan_data,
    export_network_graph,
    export_correlation_matrix,
    export_metadata,
)
import json

## Step 1: Load and filter all disorders
This loads each dataset from HuggingFace, normalizes columns, and filters to SNPs with p < 1e-5.

In [ ]:
disorders = {}
failed = []

for d in DISORDERS:
    print(f"Loading {d.name} ({d.hf_id}, config={d.config})...")
    try:
        df = load_disorder(d, p_threshold=1e-5)
        disorders[d.name] = df
        print(f"  \u2713 {len(df):,} significant SNPs")
    except Exception as e:
        failed.append((d.name, str(e)))
        print(f"  \u2717 Failed: {e}")

print(f"\nLoaded {len(disorders)}/{len(DISORDERS)} disorders")
if failed:
    print(f"Failed: {failed}")

## Step 2: Compute cross-disorder correlation matrix
Uses Pearson correlation of effect sizes (beta) on shared SNPs as a proxy for genetic correlation.

In [ ]:
correlation = compute_correlation_matrix(disorders)
print("Correlation matrix computed:")
for i, label in enumerate(correlation["labels"]):
    row = [f"{v:+.2f}" for v in correlation["values"][i]]
    print(f"  {label:20s} {' '.join(row)}")

## Step 3: Export JSON snapshots

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
colors = {d.name: d.color for d in DISORDERS}

# Export all snapshots
export_correlation_matrix(correlation, DATA_DIR)
print("\u2713 Correlation matrix exported")

export_manhattan_data(disorders, DATA_DIR)
print("\u2713 Manhattan data exported")

sig_counts = {name: len(df) for name, df in disorders.items()}
graph = export_network_graph(correlation, sig_counts, colors)
(DATA_DIR / "network_graph.json").write_text(json.dumps(graph))
print("\u2713 Network graph exported")

export_metadata(disorders, colors, DATA_DIR)
print("\u2713 Metadata exported")

print(f"\nAll snapshots written to {DATA_DIR}")